# EDA 003.02 — Mutual Information

**Kaggle Feature Engineering Course — Lesson 2**
**Reference:** [Mutual Information](https://www.kaggle.com/code/ryanholbrook/mutual-information) by Ryan Holbrook

---

## Learning Objectives

By the end of this notebook you will understand:

1. What **Mutual Information (MI)** measures and how it differs from correlation
2. Why MI is a better **feature relevance score** than Pearson correlation
3. How to compute MI for **regression and classification** tasks using `sklearn`
4. How to use MI scores to **rank and select features**
5. The **limitations** of MI as a feature selection tool

---

## What Is Mutual Information?

> **Mutual Information** measures how much knowing the value of one variable reduces uncertainty about another.

- MI = 0 → the two variables are **completely independent** (knowing X tells you nothing about Y)
- MI > 0 → knowing X **reduces uncertainty** about Y (the higher, the more useful)

### MI vs Pearson Correlation

| Property | Correlation | Mutual Information |
|---|---|---|
| Captures linear relationships | ✅ | ✅ |
| Captures non-linear relationships | ❌ | ✅ |
| Range | -1 to +1 | 0 to ∞ |
| Detects direction (positive/negative) | ✅ | ❌ (just strength) |
| Works with categorical features | Limited | ✅ |

**Key insight:** A feature can have near-zero correlation with the target but high MI (e.g. a U-shaped relationship).

---

## The Mathematics (Intuition)

$$MI(X; Y) = \sum_{x,y} P(x, y) \log \frac{P(x,y)}{P(x)\,P(y)}$$

Think of it as comparing the **joint distribution** $P(x,y)$ to what we'd expect if X and Y were independent $P(x)P(y)$.  
When they match → MI = 0. When they diverge → MI > 0.

---

## sklearn API

```python
from sklearn.feature_selection import mutual_info_regression   # for continuous targets
from sklearn.feature_selection import mutual_info_classif      # for categorical targets
```

Both return an array of MI scores — one per feature.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import mutual_info_regression, mutual_info_classif

sns.set_theme(style="whitegrid")
%matplotlib inline

---
## Demo 1 — MI Detects Non-linear Relationships That Correlation Misses

We create 4 features with different relationships to the target:
- **Linear** — classic linear relationship
- **Quadratic** — non-linear, symmetric (correlation ≈ 0!)
- **Noisy linear** — weak linear signal
- **Independent** — no relationship at all

In [ ]:
np.random.seed(42)
n = 1000
x_linear    = np.random.uniform(-3, 3, n)
x_quadratic = np.random.uniform(-3, 3, n)
x_noisy     = np.random.uniform(-3, 3, n)
x_noise     = np.random.uniform(-3, 3, n)

y = (2 * x_linear          # strong linear contribution
     + x_quadratic**2       # quadratic contribution
     + 0.5 * x_noisy        # weak linear
     + np.random.normal(0, 0.5, n))

X = pd.DataFrame({
    'linear':    x_linear,
    'quadratic': x_quadratic,
    'noisy':     x_noisy,
    'random':    x_noise,
})

# Pearson correlation with target
pearson = X.corrwith(pd.Series(y, name='target')).abs().rename('|Correlation|')

# Mutual information
mi_scores = mutual_info_regression(X, y, random_state=42)
mi = pd.Series(mi_scores, index=X.columns, name='Mutual Info')

comparison = pd.DataFrame({'|Correlation|': pearson, 'Mutual Info': mi})
print(comparison.round(4))

# Plot side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
comparison['|Correlation|'].sort_values().plot(kind='barh', ax=axes[0],
    color='steelblue', title='Pearson |Correlation| with Target')
comparison['Mutual Info'].sort_values().plot(kind='barh', ax=axes[1],
    color='darkorange', title='Mutual Information with Target')
plt.tight_layout()
plt.show()

**Observe:** The `quadratic` feature has near-zero Pearson correlation (symmetric relationship cancels out) but **high MI** — MI correctly identifies it as highly informative.

---
## Demo 2 — MI as a Feature Ranking Tool

A common workflow: compute MI scores for all features, rank them, and focus engineering effort on top-ranked features (or prune low-MI features).

In [ ]:
def make_mi_scores(X, y, task='regression'):
    """Compute and return MI scores as a sorted DataFrame."""
    # Encode any categorical columns as integer codes
    X_enc = X.copy()
    for col in X_enc.select_dtypes('object'):
        X_enc[col] = X_enc[col].astype('category').cat.codes

    if task == 'regression':
        scores = mutual_info_regression(X_enc, y, random_state=42)
    else:
        scores = mutual_info_classif(X_enc, y, random_state=42)

    mi = pd.Series(scores, index=X.columns).sort_values(ascending=False)
    return mi

def plot_mi_scores(mi, title='Mutual Information Scores'):
    fig, ax = plt.subplots(figsize=(8, max(3, len(mi) * 0.4)))
    mi.sort_values().plot(kind='barh', ax=ax, color='darkorange', edgecolor='white')
    ax.set_xlabel('Mutual Information')
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


# --- Simulate a richer dataset ---
np.random.seed(0)
n = 500
age         = np.random.randint(18, 70, n)
salary      = age * 800 + np.random.normal(0, 5000, n)
experience  = np.maximum(0, age - 22 + np.random.randint(-3, 3, n))
education   = np.random.choice(['high_school', 'bachelors', 'masters', 'phd'], n,
                                p=[0.3, 0.4, 0.2, 0.1])
noise1      = np.random.normal(0, 1, n)
noise2      = np.random.choice(['A', 'B', 'C'], n)

# Target: performance score — driven by salary, experience, education
edu_bonus   = {'high_school': 0, 'bachelors': 5, 'masters': 10, 'phd': 15}
target      = (salary / 5000 + experience * 0.3 +
               pd.Series(education).map(edu_bonus).values +
               np.random.normal(0, 2, n))

X = pd.DataFrame({'age': age, 'salary': salary, 'experience': experience,
                  'education': education, 'noise1': noise1, 'noise2': noise2})

mi = make_mi_scores(X, target, task='regression')
print(mi.round(4))
plot_mi_scores(mi, 'MI Scores — Feature Ranking for Performance Target')

---
## Demo 3 — MI for Classification

MI works equally well for categorical targets using `mutual_info_classif`.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
X_bc = pd.DataFrame(data.data, columns=data.feature_names)
y_bc = data.target

mi_classif = make_mi_scores(X_bc, y_bc, task='classification')
print("Top 10 features by MI (Breast Cancer dataset):")
print(mi_classif.head(10).round(4))
plot_mi_scores(mi_classif.head(10), 'Top 10 MI Scores — Breast Cancer Classification')

---
## Limitations of Mutual Information

- MI scores features **individually** — it doesn't account for **redundancy** between features (two highly correlated features may both score high, but only one is needed)
- MI = 0 doesn't mean a feature is **useless** — it could become useful in combination with other features
- MI is **sensitive to the number of samples** and can be noisy on small datasets
- MI doesn't reveal **direction** (positive vs negative influence)

**Use MI as a starting point for exploration, not a final verdict.**

---

## Key Takeaways

- MI measures **statistical dependence** — it captures all relationships, not just linear ones
- Use `mutual_info_regression` for continuous targets, `mutual_info_classif` for categorical
- MI is best used as a **feature ranking tool** to guide where to invest engineering effort
- Always visualise the relationship of top-MI features with the target to understand *how* they're related

---

## Further Reading

- [Kaggle Tutorial — Mutual Information](https://www.kaggle.com/code/ryanholbrook/mutual-information)
- [sklearn docs — `mutual_info_regression`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.mutual_info_regression.html)
- [Information Theory, Inference and Learning Algorithms](https://www.inference.org.uk/itprnn/book.pdf) — David MacKay (free PDF, Ch. 8)
- [Feature Selection with Mutual Information](https://machinelearningmastery.com/information-gain-and-mutual-information/) — ML Mastery